# 🚀 Giorno 2 - Lab 3: Esercitazione Hands-On sul proprio Project Canvas

Nel primo giorno di corso hai compilato il tuo **Project Canvas** per delineare il tuo caso d'uso agentico personale.
Nel Giorno 2 abbiamo approfondito:
- **I Protocolli di comunicazione standardizzata**: MCP e A2A.
- **La distinzione fondamentale**: Architettura vs Protocollo vs Framework.
- **La formalizzazione dei Tool**: Schemi JSON, input, output e gestione degli errori.

---
### 🎯 Obiettivi di questo Laboratorio
In questo notebook trasformerai i **Blocchi 5 (Dati & Fonti) e 6 (Tool & Permessi)** del tuo Project Canvas in una specifica tecnica funzionante con **Ollama (`llama3.2`)**:
1. 📝 Definire gli schemi formali JSON per i **Tool del tuo progetto personale**.
2. 💻 Implementare i mockup in Python dei tuoi tool.
3. 🤖 Testare il comportamento agentico di **Llama 3.2** in esecuzione locale sul tuo laptop.
4. 📄 Definire la **Agent Card A2A** del tuo agente.

## 1. Il tuo Caso d'Uso: Sintesi dal Project Canvas
Compila la cella di testo sottostante con i dettagli del tuo progetto personale (puoi prendere spunto dal Canvas del Giorno 1).

### 📋 Scheda del Mio Progetto:
- **Nome Studente**: *[Il tuo nome]*
- **Titolo Progetto**: *[Nome del tuo caso d'uso]*
- **Obiettivo dell'Agente**: *[Cosa deve risolvere l'agente]*
- **Attori Coinvolti**: *[Utente, altri sistemi/agenti]*
- **Fonti Dati Principali**: *[Es: database clienti, documentazione policy, API meteo, ecc.]*

## 2. Definizione dei Tool per il tuo Progetto (Blocco 6 del Canvas)
Scegli **2 o 3 tool chiave** necessari al tuo agente.
Per ciascun tool definisci:
- Nome (`name`)
- Descrizione in linguaggio naturale per il modello (`description`)
- Parametri di input con tipi e campi obbligatori (`parameters`)

In [ ]:
# DEFINISCI QUI I TOOL DEL TUO PROGETTO PERSONALE
# Esempio: Modifica o sostituisci con i tool del tuo caso d'uso

my_project_tools = [
    {
        "type": "function",
        "function": {
            "name": "lookup_user_profile",  # Modifica con il tuo tool 1
            "description": "Recupera i dati e i permessi dell'utente dal database aziendale.",
            "parameters": {
                "type": "object",
                "properties": {
                    "user_id": {
                        "type": "string",
                        "description": "ID univoco dell'utente"
                    }
                },
                "required": ["user_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "check_service_availability",  # Modifica con il tuo tool 2
            "description": "Verifica la disponibilità di una risorsa o servizio.",
            "parameters": {
                "type": "object",
                "properties": {
                    "resource_name": {
                        "type": "string",
                        "description": "Nome del servizio o risorsa richiesta"
                    },
                    "date": {
                        "type": "string",
                        "description": "Data richiesta nel formato YYYY-MM-DD"
                    }
                },
                "required": ["resource_name"]
            }
        }
    }
]

print(f"Definiti {len(my_project_tools)} tool per il progetto personale.")

## 3. Implementazione Mock dei tuoi Tool in Python
Scrivi la logica Python che simula il comportamento dei tool (lettura dati da dizionari mock o calcoli).

In [ ]:
def execute_my_tool(name: str, args: dict) -> dict:
    """
    Router di esecuzione per i tool del tuo progetto.
    """
    print(f"🔧 Esecuzione tool: {name} con argomenti: {args}")
    
    # Tool 1: personalizza qui
    if name == "lookup_user_profile":
        user_id = args.get("user_id")
        return {
            "result": {
                "user_id": user_id,
                "status": "ACTIVE",
                "tier": "PREMIUM",
                "credits": 250
            }
        }
        
    # Tool 2: personalizza qui
    elif name == "check_service_availability":
        resource = args.get("resource_name")
        return {
            "result": {
                "resource": resource,
                "available": True,
                "slots": ["10:00", "14:30", "16:00"]
            }
        }
        
    # Gestione tool sconosciuto
    return {"error": {"code": "NOT_FOUND", "message": f"Tool '{name}' non implementato."}}

# Test rapido di esecuzione locale
print("Test tool 1:", execute_my_tool("lookup_user_profile", {"user_id": "USR-101"}))
print("Test tool 2:", execute_my_tool("check_service_availability", {"resource_name": "Server-GPU-01"}))

## 4. Test con Ollama (`llama3.2`): L'Agente del tuo Progetto in Azione!
Testiamo il tuo agente facendogli elaborare una richiesta reale con Tool Calling.

In [ ]:
import requests
import json

OLLAMA_URL = "http://localhost:11434/api/chat"
MODEL_NAME = "llama3.2"

def test_my_custom_agent(user_query: str, system_instructions: str):
    messages = [
        {"role": "system", "content": system_instructions},
        {"role": "user", "content": user_query}
    ]
    
    print(f"👤 RICHIESTA UTENTE: {user_query}\n")
    
    # 1. Chiamata al modello con i tuoi tool
    payload = {
        "model": MODEL_NAME,
        "messages": messages,
        "tools": my_project_tools,
        "stream": False,
        "options": {"temperature": 0.0}
    }
    
    resp = requests.post(OLLAMA_URL, json=payload)
    if resp.status_code != 200:
        print("Errore Ollama:", resp.text)
        return
        
    msg = resp.json()["message"]
    tool_calls = msg.get("tool_calls", [])
    
    messages.append(msg)
    
    if not tool_calls:
        print("🤖 Risposta diretta del modello:")
        print(msg.get("content", ""))
        return
        
    for call in tool_calls:
        fn_name = call["function"]["name"]
        fn_args = call["function"]["arguments"]
        print(f"👉 Tool invocato da Llama 3.2: {fn_name}")
        print(f"   Parametri: {fn_args}")
        
        # Esecuzione
        tool_result = execute_my_tool(fn_name, fn_args)
        messages.append({
            "role": "tool",
            "name": fn_name,
            "content": json.dumps(tool_result, ensure_ascii=False)
        })
        
    # 2. Seconda chiamata per sintesi finale
    final_payload = {
        "model": MODEL_NAME,
        "messages": messages,
        "stream": False
    }
    final_resp = requests.post(OLLAMA_URL, json=final_payload)
    print("\n🤖 RISPOSTA FINALE GENERATA:")
    print(final_resp.json()["message"]["content"])

# Personalizza il prompt di sistema e la query per il tuo progetto
my_system_prompt = (
    "Sei l'assistente dedicato per il sistema aziendale. "
    "Utilizza sempre i tool a disposizione per verificare i dati prima di formulare risposte definitive."
)
my_user_test_query = "Ciao! Sono l'utente USR-101. Vorrei sapere se la risorsa Server-GPU-01 è disponibile oggi."

test_my_custom_agent(my_user_test_query, my_system_prompt)

## 5. Progetta l'Agent Card A2A per il tuo Agente
Se il tuo agente dovesse collaborare con agenti esterni (es. agenti dei clienti o di altri dipartimenti), come si presenterebbe la sua **Agent Card**?

Compila il template sottostante:

In [ ]:
my_agent_card = {
    "a2a_version": "1.0",
    "agent_id": "agent.custom.myproject",  # Modifica con l'id del tuo agente
    "name": "Assistente Personalizzato Progetto",
    "description": "Descrizione chiara delle capacità dell'agente...",
    "endpoint": "https://api.myproject.corp/a2a/v1",
    "skills": [
        {
            "id": "skill_principale_1",
            "name": "Nome Skill 1",
            "description": "Cosa fa questa skill e quando chiamarla",
            "input_schema": {"param1": "string"},
            "output_schema": {"result": "object"}
        }
    ]
}

print("📋 La mia Agent Card A2A:")
print(json.dumps(my_agent_card, indent=2, ensure_ascii=False))

---
### 🎉 Congratulazioni! Hai completato il Laboratorio del Giorno 2!
Hai formalizzato:
- I tuoi Tool con **JSON Schema standard MCP**.
- Il test operativo locale con **`llama3.2`**.
- La struttura di cooperazione multi-agente con **A2A**.

**Prossimo passo (Giorno 3)**: Importeremo questi tool e workflow all'interno dell'ambiente visuale **Langflow** per costruire pipeline interattive complete!